# Mini Project Starter — Reliable Retail Assistant (Text RAG + Multimodal RAG)

## Goal
Build a retail assistant that supports:
- **Text RAG**: text query → retrieve products → recommend (grounded)
- **Multimodal RAG**: image → extract query → retrieve products → recommend

## Must-have engineering qualities
- Structured output schema (JSON) + validation
- Budgeted agent steps
- Observability logs
- Lightweight evaluation harness (regression checks)

> You are expected to fill all TODO sections.

In [0]:
# !pip -q install --upgrade openai faiss-cpu pandas numpy pydantic pillow

## Setup: Vocareum OpenAI Gateway

Set your API key as an environment variable before continuing.

Example:
```python
import os
os.environ["OPENAI_API_KEY"] = "voc-...."

In [0]:

---

### **Cell 4 (Code) — Imports + client**
```python
import os, json, time, base64, mimetypes
import numpy as np
import pandas as pd
import faiss

from typing import List, Dict, Any, Optional, Literal
from pydantic import BaseModel, Field, ValidationError
from PIL import Image

from openai import OpenAI

BASE_URL = "https://openai.vocareum.com/v1"
API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY env var before running.")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

TEXT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"
VISION_MODEL = "gpt-4o-mini"   # may or may not support vision in your policy

## Step 1 — Create a Product Catalog

Minimum: **20 products**
Recommended: **30–50 products**

Each product must include:
- id, name, category, brand, price, specs, desc

TODO: Expand the dataset to cover multiple categories like:
- laptops, phones, shoes, headphones/earbuds

Tip: Keep specs/desc realistic — retrieval quality depends on it.

In [0]:
products = [
    # TODO: Add more products across categories (target 30-50)

    {"id":"S1","name":"Nike Air Zoom Pegasus","category":"shoes","price":95,"brand":"Nike",
     "specs":"Use: Running, Cushioning: Medium, Weight: Light",
     "desc":"Daily trainer for road running, balanced cushioning."},

    {"id":"S3","name":"Converse Chuck Taylor","category":"shoes","price":55,"brand":"Converse",
     "specs":"Use: Casual, Cushioning: Low, Weight: Light",
     "desc":"Classic casual sneaker, not designed for running."},

    {"id":"P2","name":"Google Pixel 7a","category":"phone","price":499,"brand":"Google",
     "specs":"Wireless charging: Yes, 8GB RAM, 128GB storage, 5G",
     "desc":"Great camera and clean Android, good value with wireless charging."},

    {"id":"P3","name":"Moto G Power","category":"phone","price":199,"brand":"Motorola",
     "specs":"Wireless charging: No, 4GB RAM, 64GB storage, 4G",
     "desc":"Budget phone with big battery, basic performance."},

    {"id":"L1","name":"Acer Nitro 5","category":"laptop","price":899,"brand":"Acer",
     "specs":"i7, RTX 4060, 16GB RAM, 512GB SSD, 144Hz",
     "desc":"Gaming laptop with dedicated GPU, high refresh display."},
]

df = pd.DataFrame(products)
df.head()

## Step 2 — Build Retrieval Documents

We will convert each product row into a clean text document.
This is what gets embedded and indexed.

TODO:
- Implement `product_to_text(row)` properly.
- Add a `doc` column to df.

In [0]:
def product_to_text(row: pd.Series) -> str:
    # TODO: Implement a good document string with all important fields
    # Must include: id, name, category, brand, price, specs, desc
    return (
        f"Product ID: {row['id']}\n"
        f"Name: {row['name']}\n"
        f"Category: {row['category']}\n"
        f"Brand: {row['brand']}\n"
        f"Price: ${row['price']}\n"
        f"Specs: {row['specs']}\n"
        f"Description: {row['desc']}\n"
    )

df["doc"] = df.apply(product_to_text, axis=1)
df["doc"].iloc[0]

## Step 3 — Build Embeddings + FAISS Index

TODO:
- Implement `embed_texts(texts)`
- Build FAISS index using cosine similarity:
  - L2 normalize vectors
  - use IndexFlatIP
- Add all product vectors to the index

In [0]:
def embed_texts(texts: List[str], model: str = EMBED_MODEL) -> np.ndarray:
    # TODO: Implement embeddings call using client.embeddings.create
    resp = client.embeddings.create(model=model, input=texts)
    vecs = [d.embedding for d in resp.data]
    return np.array(vecs, dtype=np.float32)

In [0]:
docs = df["doc"].tolist()
emb = embed_texts(docs, EMBED_MODEL)

dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)

# TODO: normalize and add to index
faiss.normalize_L2(emb)
index.add(emb)

print("Index ready:", index.ntotal, "vectors | dim:", dim)

## Step 4 — Retrieval Tool

TODO:
- Implement `retrieve_products(query, k)`
- It must return:
  - product id, name, category, price, doc, score

In [0]:
def retrieve_products(query: str, k: int = 5) -> List[Dict[str, Any]]:
    # TODO: Embed query, normalize, search FAISS
    qv = embed_texts([query], EMBED_MODEL)
    faiss.normalize_L2(qv)

    scores, ids = index.search(qv, k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        row = df.iloc[int(idx)]
        results.append({
            "id": row["id"],
            "name": row["name"],
            "category": row["category"],
            "price": float(row["price"]),
            "doc": row["doc"],
            "score": float(score)
        })
    return results

retrieve_products("running shoes under 100", k=3)

## Step 5 — Structured Output Schema

Your model must output strict JSON matching a schema.

TODO:
- Define Pydantic models:
  - RecommendedProduct
  - RetailAssistantOutput
- Implement strict JSON parse + validation

In [0]:
class RecommendedProduct(BaseModel):
    product_id: str
    product_name: str
    price: float
    reason: str
    citations: List[str] = Field(default_factory=list)

class RetailAssistantOutput(BaseModel):
    query_type: Literal["recommendation","comparison","qa"]
    user_need: str
    recommended_products: List[RecommendedProduct]

def parse_and_validate_output(raw: str) -> RetailAssistantOutput:
    # TODO: strict JSON parse + schema validation
    data = json.loads(raw)
    return RetailAssistantOutput(**data)

## Step 6 — Recommendation Tool (Grounded Generation)

TODO:
- Call the LLM with:
  - user_query
  - retrieved product docs ONLY
- Force JSON only output
- Add citations by product IDs

In [0]:
def tool_recommend(user_query: str, retrieved: List[Dict[str, Any]]) -> str:
    context = "\n---\n".join([r["doc"] for r in retrieved])

    prompt = f"""
You are a retail assistant. Recommend up to 3 products for the user query.
Use ONLY the provided product context. Do not invent products.

Return STRICT JSON with:
{{
  "query_type": "recommendation|comparison|qa",
  "user_need": "<short>",
  "recommended_products": [
    {{
      "product_id": "<id>",
      "product_name": "<name>",
      "price": <number>,
      "reason": "<1-2 lines grounded in specs/desc>",
      "citations": ["<product_id>", ...]
    }}
  ]
}}

User query: {user_query}

Product context:
{context}
""".strip()

    resp = client.chat.completions.create(
        model=TEXT_MODEL,
        messages=[
            {"role":"system","content":"Return valid JSON only. No markdown. No extra keys."},
            {"role":"user","content":prompt}
        ],
        temperature=0.2
    )
    return resp.choices[0].message.content

## Step 7 — Observability Logs

TODO:
- Implement `log_event(logs, step, tool, payload)`
- The agent must log:
  - image extraction output (if used)
  - retrieval query + top ids + scores
  - model output length
  - validation pass/fail

In [0]:
def log_event(logs: List[Dict[str, Any]], step: int, tool: str, payload: Dict[str, Any]):
    logs.append({
        "ts": time.time(),
        "step": step,
        "tool": tool,
        "payload": payload
    })

## Step 8 — Multimodal RAG: Image → Query Extraction

We do NOT embed the image directly.
Instead:
- Use a multimodal LLM to extract a search query in JSON
- Use that extracted query for text retrieval

TODO:
- Implement:
  - image_to_data_url(image_path)
  - ImageQueryExtraction schema
  - extract_query_from_image(image_path) returning structured object

Important:
- If vision is not allowed by policy, you must fail gracefully
  OR provide a fallback (manual caption input).

In [0]:
def image_to_data_url(image_path: str) -> str:
    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/png"
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime_type};base64,{b64}"

In [0]:
class ImageQueryExtraction(BaseModel):
    category_guess: Optional[Literal["laptop","phone","headphones","earbuds","shoes","unknown"]] = "unknown"
    brand_guess: Optional[str] = None
    key_attributes: List[str] = Field(default_factory=list)
    search_query: str

In [0]:
def extract_query_from_image(image_path: str) -> ImageQueryExtraction:
    img_url = image_to_data_url(image_path)

    prompt = """
Given an image of a product, extract a retrieval query.

Return STRICT JSON with keys:
- category_guess: ["laptop","phone","headphones","earbuds","shoes","unknown"]
- brand_guess: string or null
- key_attributes: list of short phrases
- search_query: short semantic search query (category + key hints)

Rules:
- Avoid hallucinating specs.
- Be conservative.
""".strip()

    resp = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {"role":"system","content":"Return valid JSON only. No markdown."},
            {"role":"user","content":[
                {"type":"text","text":prompt},
                {"type":"image_url","image_url":{"url":img_url}}
            ]}
        ],
        temperature=0.2
    )

    raw = resp.choices[0].message.content
    data = json.loads(raw)
    return ImageQueryExtraction(**data)

## Step 9 — Unified Agent Controller (Text RAG or Multimodal RAG)

TODO:
- Implement `build_retrieval_query(user_query, image_extraction)`
- Implement `run_retail_agent(user_query=None, image_path=None)`
  - enforce max_steps
  - logs for each step
  - retrieval + recommend + validate
  - if image extraction fails and no text query exists → return a clean error

In [0]:
def build_retrieval_query(user_query: Optional[str], image_extraction: Optional[ImageQueryExtraction]) -> str:
    parts = []
    if user_query and user_query.strip():
        parts.append(user_query.strip())
    if image_extraction:
        parts.append(image_extraction.search_query.strip())
        if image_extraction.brand_guess:
            parts.append(f"brand {image_extraction.brand_guess}")
        if image_extraction.key_attributes:
            parts.append(" ".join(image_extraction.key_attributes))
    return " | ".join(parts) if parts else ""

In [0]:
def run_retail_agent(user_query: Optional[str] = None,
                     image_path: Optional[str] = None,
                     k: int = 5,
                     max_steps: int = 6) -> Dict[str, Any]:

    if (not user_query or not user_query.strip()) and (not image_path):
        raise ValueError("Provide either a text query or an image_path.")

    logs: List[Dict[str, Any]] = []
    step = 0
    image_extraction = None

    # Step 1: image extraction if image provided
    if image_path:
        step += 1
        try:
            image_extraction = extract_query_from_image(image_path)
            log_event(logs, step, "image_extract_query", image_extraction.model_dump())
        except Exception as e:
            log_event(logs, step, "image_extract_query_failed", {"error": str(e)})
            if not user_query:
                return {"error":"vision_not_available", "message":"Image mode not available. Use text query.", "logs": logs}

    # Step 2: retrieval
    step += 1
    retrieval_query = build_retrieval_query(user_query, image_extraction)
    retrieved = retrieve_products(retrieval_query, k=k)
    log_event(logs, step, "retrieve_products", {"retrieval_query": retrieval_query, "top_ids":[r["id"] for r in retrieved]})

    # Step 3: recommend
    step += 1
    effective_query = user_query.strip() if user_query and user_query.strip() else "(image-based query)"
    raw_json = tool_recommend(effective_query, retrieved)
    log_event(logs, step, "recommend", {"raw_len": len(raw_json)})

    # Step 4: validate
    step += 1
    parsed = parse_and_validate_output(raw_json)
    log_event(logs, step, "finish", {"status":"ok", "n_recs": len(parsed.recommended_products)})

    meta = {
        "mode": "multimodal_rag" if image_path else "text_rag",
        "retrieval_query": retrieval_query,
        "retrieved_top_ids": [r["id"] for r in retrieved],
    }

    return {
        "output": parsed.model_dump(),
        "meta": meta,
        "retrieved": [{"id": r["id"], "score": r["score"], "name": r["name"]} for r in retrieved],
        "image_extraction": image_extraction.model_dump() if image_extraction else None,
        "logs": logs
    }

## Step 10 — Quick Manual Tests

TODO:
- Run at least 3 text queries
- Run 1 image query (if supported)
- Print output + meta + logs

In [0]:
print(json.dumps(run_retail_agent(user_query="Recommend running shoes under $100", k=5), indent=2)[:1500])
print(json.dumps(run_retail_agent(user_query="Which phone supports wireless charging under $600?", k=5), indent=2)[:1500])
print(json.dumps(run_retail_agent(user_query="Suggest a gaming laptop under $1000", k=5), indent=2)[:1500])

# Optional image test
# IMAGE_PATH = "shoe.png"
# print(json.dumps(run_retail_agent(image_path=IMAGE_PATH, k=5), indent=2)[:2000])

In [0]:
## Step 11 — Evaluation Harness (Regression Checks)

TODO:
- Create at least 5 evaluation cases
- Each case should specify:
  - query
  - must_include_any_ids
  - must_not_include_ids
- Compute pass rate
- Add an assert threshold (like >= 0.6)

In [0]:
eval_cases = [
    # TODO: Add more cases (at least 5)
    {
        "query": "Recommend running shoes under $100 for daily road running.",
        "must_include_any_ids": ["S1"],
        "must_not_include_ids": ["S3"],
    },
    {
        "query": "Which phone supports wireless charging under $600?",
        "must_include_any_ids": ["P2"],
        "must_not_include_ids": ["P3"],
    },
]

def evaluate_case(case: Dict[str, Any]) -> Dict[str, Any]:
    res = run_retail_agent(user_query=case["query"], k=5)
    rec_ids = [p["product_id"] for p in res["output"]["recommended_products"]]

    ok_include = any(i in rec_ids for i in case["must_include_any_ids"]) if case["must_include_any_ids"] else True
    ok_exclude = all(i not in rec_ids for i in case["must_not_include_ids"])

    return {
        "query": case["query"],
        "rec_ids": rec_ids,
        "pass_include": ok_include,
        "pass_exclude": ok_exclude,
        "pass": bool(ok_include and ok_exclude),
    }

results = [evaluate_case(c) for c in eval_cases]
pd.DataFrame(results)

In [0]:
df_eval = pd.DataFrame(results)
acc = float(df_eval["pass"].mean())
print("Pass rate:", round(acc*100, 1), "%")

# TODO: choose your threshold
assert acc >= 0.6, "Regression: pass rate below threshold"
print("✅ Regression check passed")

# Submission Checklist

✅ Product dataset: 20–50 products  
✅ FAISS index built over product docs  
✅ retrieve_products(query,k) works  
✅ LLM recommendation returns strict JSON  
✅ Schema validation implemented  
✅ Unified agent supports text or image input  
✅ Logs exist for each run  
✅ Evaluation harness has ≥5 cases + assert threshold  

**Deliverables**
- Notebook (.ipynb)
- Short report (2–4 pages)
- Example outputs (text + image if supported)